In [1]:
print("Hello World")

Hello World


In [2]:
from dotenv import load_dotenv
load_dotenv()


True

In [9]:
import pandas as pd
# QA # QA
inputs = [
    "How does Multimodal RAG (M-RAG) handle an image of a complex technical chart?",
    "What is the 'Context-Generation Hallucination' risk in multimodal RAG systems?",
    "In the MMed-RAG paper, how much did factual accuracy improve for medical Vision-Language Models (LVLMs)?",
    "Why is fine-tuning often preferred for domain-specific layout understanding (like reading invoices)?",
    "How does RAG compare to Fine-Tuning when identifying a rare medical condition from a single X-ray?",
    "What happens to M-RAG performance when the retrieved image is low-resolution or has OCR errors?",
    "Can fine-tuning a Vision-Language Model help it ignore 'distractor' images in a retrieval set?",
    "What is the 'Cumulative Benefit' of combining RAG with preference fine-tuning (RAG-PT)?",
    "Which approach is better for a company that adds 1,000 new product photos to their catalog daily?",
    "How does M-RAG handle queries that require 'compositional reasoning' across a table and a diagram?"
]

outputs = [
    "M-RAG retrieves the image and uses a Vision-Language Model (VLM) to describe its features in the context of the query.",
    "It occurs when the VLM generates an incorrect text description of a retrieved image, misleading the final answer.",
    "MMed-RAG achieved an average improvement of 43.8% in factual accuracy across medical datasets.",
    "Fine-tuning allows the model to learn the specific spatial patterns and 'jargon' of layouts better than zero-shot retrieval.",
    "RAG is superior here; if the rare case exists in the database, RAG finds it. FT often fails to memorize 'one-shot' facts.",
    "Accuracy drops significantly; M-RAG is highly sensitive to the quality of the 'raw' multimodal context retrieved.",
    "Yes, fine-tuning on preference data (DPO/RLHF) helps the model learn which visual cues are relevant and which are noise.",
    "It aligns the model's 'eyes' with the retrieved context, reducing the chance of the model ignoring the provided image.",
    "RAG is the only viable choice; fine-tuning every day on new images is computationally and financially prohibitive.",
    "Advanced M-RAG uses 'iterative retrieval' to first parse the table, then use those insights to query the diagram."
]

# Dataset
qa_pairs = [{"question": q, "answer": a} for q, a in zip(inputs, outputs)]
df = pd.DataFrame(qa_pairs)

# Write to csv
csv_path = "C:/Users/Kethavath/multi-doc-chat/data/goldens.csv"

df.to_csv(csv_path, index=False)

In [25]:
from langsmith import Client

client = Client()
dataset_name = "MultiModal-RAG"

# Store
dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Input and expected output pairs for MultiModal Rag paper",
)
client.create_examples(
    inputs=[{"question": q} for q in inputs],
    outputs=[{"answer": a} for a in outputs],
    dataset_id=dataset.id,
)

{'example_ids': ['865d0166-7a56-4518-ac5a-53f271add5b0',
  '7b75303f-299b-43b4-85a8-a60629c5bd56',
  '0774b433-0763-4d87-b610-a10a6abc958f',
  '9f486f51-14eb-4f2b-906c-e95599645b24',
  '52702863-26cf-42e7-9ba3-e5eb1940e4c4',
  '623e4fa9-bc8a-48ea-9028-c51b21203246',
  '9c6ebc8a-8bce-47df-97e0-698e27a4ff61',
  '8360fbc7-4d49-4ad8-ba85-6a0cffaabc85',
  '91adacdf-d96b-4aad-bb37-2f16b8863585',
  '8250fd16-75df-4c4d-a994-7560c53d5268'],
 'count': 10,
 'as_of': '2026-02-19T14:40:47.384780553Z'}

In [26]:
import sys
sys.path.append("C:/Users/Kethavath/multi-doc-chat")

from pathlib import Path
from multi_doc_chat.src.document_ingestion.data_ingestion import ChatIngestor
from multi_doc_chat.src.document_chat.retrieval import ConversationalRAG
import os

# Simple file adapter for local file paths
class LocalFileAdapter:
    """Adapter for local file paths to work with ChatIngestor."""
    def __init__(self, file_path: str):
        self.path = Path(file_path)
        self.name = self.path.name
    
    def getbuffer(self) -> bytes:
        return self.path.read_bytes()


def answer_ai_report_question(
    inputs: dict,
    data_path: str = "C:\\Users\\Kethavath\\multi-doc-chat\\data\\Multimodal_RAG.pdf",
    chunk_size: int = 1000,
    chunk_overlap: int = 200,
    k: int = 5
) -> dict:
    """
    Answer questions about the Paper MultiModal RAG.
    
    Args:
        inputs: Dictionary containing the question, e.g., {"question": "What is RAG?"}
        data_path: Path to the MultiModal RAG pdf file
        chunk_size: Size of text chunks for splitting
        chunk_overlap: Overlap between chunks
        k: Number of documents to retrieve
    
    Returns:
        Dictionary with the answer, e.g., {"answer": "RAG stands for..."}
    """
    try:
        # Extract question from inputs
        question = inputs.get("question", "")
        if not question:
            return {"answer": "No question provided"}
        
        # Check if file exists
        if not Path(data_path).exists():
            return {"answer": f"Data file not found: {data_path}"}
        
        # Create file adapter
        file_adapter = LocalFileAdapter(data_path)
        
        # Build index using ChatIngestor
        ingestor = ChatIngestor(
            temp_base="data",
            faiss_base="faiss_index",
            use_session_dirs=True
        )
        
        # Build retriever
        ingestor.built_retriver(
            uploaded_files=[file_adapter],
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            k=k
        )
        
        # Get session ID and index path
        session_id = ingestor.session_id
        index_path = f"faiss_index/{session_id}"
        
        # Create RAG instance and load retriever
        rag = ConversationalRAG(session_id=session_id)
        rag.load_retriever_from_faiss(
            index_path=index_path,
            k=k,
            index_name=os.getenv("FAISS_INDEX_NAME", "index")
        )
        
        # Get answer
        answer = rag.invoke(question, chat_history=[])
        
        return {"answer": answer}
        
    except Exception as e:
        return {"answer": f"Error: {str(e)}"}


In [13]:
# Test the function with a sample question
test_input = {"question": "What is the 'Context-Generation Hallucination' risk in multimodal RAG systems?"}
result = answer_ai_report_question(test_input)
print("Question:", test_input["question"])
print("\nAnswer:", result["answer"])


{"timestamp": "2026-02-19T14:01:37.342895Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2026-02-19T14:01:37.344895Z", "level": "info", "event": "Loaded GROQ_API_KEY from individual env var"}
{"timestamp": "2026-02-19T14:01:37.345895Z", "level": "info", "event": "Loaded GOOGLE_API_KEY from individual env var"}
{"keys": {"GROQ_API_KEY": "gsk_or...", "GOOGLE_API_KEY": "AIzaSy..."}, "timestamp": "2026-02-19T14:01:37.346895Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2026-02-19T14:01:37.367894Z", "level": "info", "event": "YAML config loaded"}
{"session_id": "session_20260219_193137_396955fb", "temp_dir": "data\\session_20260219_193137_396955fb", "faiss_dir": "faiss_index\\session_20260219_193137_396955fb", "sessionized": true, "timestamp": "2026-02-19T14:01:37.369896Z", "level": "info", "event": "ChatIngestor initialized"}
{"uploaded": "Multimodal_RAG.pdf", "saved_as": "data\\sess

Question: What is the 'Context-Generation Hallucination' risk in multimodal RAG systems?

Answer: I don't know.


In [22]:
from langsmith.evaluation import evaluate

In [ ]:
# Example: Test with all golden questions
print("Testing all questions from the dataset:\n")
for i, q in enumerate(inputs, 1):
    test_input = {"question": q}
    result = answer_ai_report_question(test_input)
    print(f"Q{i}: {q}")
    print(f"A{i}: {result['answer']}\n")
    print("-" * 80 + "\n")


In [29]:
from langsmith.evaluation import evaluate
from langchain_openai import ChatOpenAI
from langsmith.evaluation.llm_evaluators import CriteriaEvaluator

# Judge model
judge_llm = ChatOpenAI(model="gpt-4o-mini")

qa_evaluator = CriteriaEvaluator.from_llm(
    llm=judge_llm,
    criteria="qa",
)

# Run evaluation using our RAG function
experiment_results = evaluate(
    answer_ai_report_question,
    data="MultiModal-RAG",
    evaluators=["qa"],
    experiment_prefix="test-multimodalrag-qa-rag",
    # Experiment metadata
    metadata={
        "variant": "Multi Modal RAG Paper",
        "chunk_size": 1000,
        "chunk_overlap": 200,
        "k": 5,
    },
)

ModuleNotFoundError: No module named 'langsmith.evaluation.llm_evaluators'

## Custom Correctness Evaluator

Creating an LLM-as-a-Judge evaluator to assess semantic and factual alignment


In [30]:
from langsmith.schemas import Run, Example
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

def correctness_evaluator(run: Run, example: Example) -> dict:
    """
    Custom LLM-as-a-Judge evaluator for correctness.
    
    Correctness means how well the actual model output matches the reference output 
    in terms of factual accuracy, coverage, and meaning.
    
    Args:
        run: The Run object containing the actual outputs
        example: The Example object containing the expected outputs
    
    Returns:
        dict with 'score' (1 for correct, 0 for incorrect) and 'reasoning'
    """
    # Extract actual and expected outputs
    actual_output = run.outputs.get("answer", "")
    expected_output = example.outputs.get("answer", "")
    input_question = example.inputs.get("question", "")
    
    # Define the evaluation prompt
    eval_prompt = ChatPromptTemplate.from_messages([
        ("system", """You are an evaluator whose job is to judge correctness.

Correctness means how well the actual model output matches the reference output in terms of factual accuracy, coverage, and meaning.

- If the actual output matches the reference output semantically (even if wording differs), it should be marked correct.
- If the output misses key facts, introduces contradictions, or is factually incorrect, it should be marked incorrect.

Do not penalize for stylistic or formatting differences unless they change meaning."""),
        ("human", """<example>
<input>
{input}
</input>

<output>
Expected Output: {expected_output}

Actual Output: {actual_output}
</output>
</example>

Please grade the following agent run given the input, expected output, and actual output.
Focus only on correctness (semantic and factual alignment).

Respond with:
1. A brief reasoning (1-2 sentences)
2. A final verdict: either "CORRECT" or "INCORRECT"

Format your response as:
Reasoning: [your reasoning]
Verdict: [CORRECT or INCORRECT]""")
    ])
    
    # Initialize LLM (using Gemini as shown in your config)
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-pro",
        temperature=0
    )
    
    # Create chain and invoke
    chain = eval_prompt | llm
    
    try:
        response = chain.invoke({
            "input": input_question,
            "expected_output": expected_output,
            "actual_output": actual_output
        })
        
        response_text = response.content
        
        # Parse the response
        reasoning = ""
        verdict = ""
        
        for line in response_text.split('\n'):
            if line.startswith("Reasoning:"):
                reasoning = line.replace("Reasoning:", "").strip()
            elif line.startswith("Verdict:"):
                verdict = line.replace("Verdict:", "").strip()
        
        # Convert verdict to score (1 for correct, 0 for incorrect)
        score = 1 if "CORRECT" in verdict.upper() else 0
        
        return {
            "key": "correctness",
            "score": score,
            "reasoning": reasoning,
            "comment": f"Verdict: {verdict}"
        }
        
    except Exception as e:
        return {
            "key": "correctness",
            "score": 0,
            "reasoning": f"Error during evaluation: {str(e)}"
        }


### Run Evaluation with Custom Correctness Evaluator


In [ ]:
# Run evaluation with the custom correctness evaluator
from langsmith.evaluation import evaluate

# Define evaluators - using custom correctness evaluator
evaluators = [correctness_evaluator]

dataset_name = "MultiModal-RAG"

# Run evaluation
experiment_results = evaluate(
    answer_ai_report_question,
    data=dataset_name,
    evaluators=evaluators,
    experiment_prefix="agenticAIReport-correctness-eval",
    description="Evaluating RAG system with custom correctness evaluator (LLM-as-a-Judge)",
    metadata={
        "variant": "RAG with FAISS and AI Engineering Report",
        "evaluator": "custom_correctness_llm_judge",
        "model": "gemini-2.5-pro",
        "chunk_size": 1000,
        "chunk_overlap": 200,
        "k": 5,
    },
)

print("\nEvaluation completed! Check the LangSmith UI for detailed results.")


View the evaluation results for experiment: 'agenticAIReport-correctness-eval-3fab3148' at:
https://smith.langchain.com/o/324ea00f-d8c4-46a3-8fea-04ff36e921b7/datasets/24dc1250-22cf-4512-aef0-b4e9a294ea8c/compare?selectedSessions=aa36bbfe-2a82-46d4-8bdb-08f3340232eb




0it [00:00, ?it/s]{"timestamp": "2026-02-19T14:52:28.455653Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2026-02-19T14:52:28.456652Z", "level": "info", "event": "Loaded GROQ_API_KEY from individual env var"}
{"timestamp": "2026-02-19T14:52:28.456652Z", "level": "info", "event": "Loaded GOOGLE_API_KEY from individual env var"}
{"keys": {"GROQ_API_KEY": "gsk_or...", "GOOGLE_API_KEY": "AIzaSy..."}, "timestamp": "2026-02-19T14:52:28.457651Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2026-02-19T14:52:28.460182Z", "level": "info", "event": "YAML config loaded"}
{"session_id": "session_20260219_202228_ebddda9e", "temp_dir": "data\\session_20260219_202228_ebddda9e", "faiss_dir": "faiss_index\\session_20260219_202228_ebddda9e", "sessionized": true, "timestamp": "2026-02-19T14:52:28.462165Z", "level": "info", "event": "ChatIngestor initialized"}
{"uploaded": "Multimodal_RAG.pdf", "save

### Optional: Combine Multiple Evaluators

You can use multiple evaluators together to get different perspectives on your RAG system's performance.


In [ ]:
# Example: Combine custom correctness evaluator with LangChain's built-in evaluators
from langsmith.evaluation import evaluate, LangChainStringEvaluator

# Combine custom and built-in evaluators
combined_evaluators = [
    correctness_evaluator,  # Custom LLM-as-a-Judge
    LangChainStringEvaluator("cot_qa"),  # Chain-of-thought QA evaluator
]

# Run evaluation with multiple evaluators
# Uncomment to run:
# experiment_results_combined = evaluate(
#     answer_ai_report_question,
#     data=dataset_name,
#     evaluators=combined_evaluators,
#     experiment_prefix="agenticAIReport-multi-eval",
#     description="Evaluating RAG system with multiple evaluators",
#     metadata={
#         "variant": "RAG with FAISS",
#         "evaluators": "correctness + cot_qa",
#         "chunk_size": 1000,
#         "chunk_overlap": 200,
#         "k": 5,
#     },
# )
